In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.cluster import KMeans

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor


In [ ]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

test_ids = test['id']

train.drop(columns=['id','Row#'], inplace=True)
test.drop(columns=['id','Row#'], inplace=True)


In [ ]:
def feature_engineering(df, kmeans=None, scaler=None, qt=None, fit=False):
    d = df.copy()

    bees = ['honeybee','bumbles','andrena','osmia']
    d['total_bees'] = d[bees].sum(axis=1)
    d['bee_div'] = d[bees].std(axis=1)
    d['bee_max'] = d[bees].max(axis=1)

    d['bees_per_clone'] = d['total_bees'] / (d['clonesize'] + 1e-6)

    d['temp_range'] = d['MaxOfUpperTRange'] - d['MinOfLowerTRange']
    d['avg_temp'] = (d['AverageOfUpperTRange'] + d['AverageOfLowerTRange']) / 2

    d['rain_temp'] = d['avg_temp'] * d['RainingDays']
    d['fruit_seed_ratio'] = d['fruitmass'] / (d['seeds'] + 1e-6)

    for c in ['clonesize','fruitmass','seeds','total_bees']:
        d[f'log_{c}'] = np.log1p(d[c])

    cluster_cols = ['clonesize','total_bees','avg_temp','RainingDays']
    if fit:
        scaler = StandardScaler()
        qt = QuantileTransformer(n_quantiles=100, output_distribution='normal')
        Xs = scaler.fit_transform(d[cluster_cols])
        Xq = qt.fit_transform(Xs)

        kmeans = KMeans(n_clusters=7, n_init=30, random_state=42)
        d['cluster'] = kmeans.fit_predict(Xq)
    else:
        Xs = scaler.transform(d[cluster_cols])
        Xq = qt.transform(Xs)
        d['cluster'] = kmeans.predict(Xq)

    return d, kmeans, scaler, qt


In [ ]:
train_fe, kmeans, scaler, qt = feature_engineering(train, fit=True)
test_fe, _, _, _ = feature_engineering(test, kmeans, scaler, qt)

X = train_fe.drop(columns=['yield'])
y_raw = train_fe['yield'].values
y_log = np.log1p(y_raw)


In [ ]:
def xgb(seed):
    return XGBRegressor(
        n_estimators=9000,
        learning_rate=0.02,
        max_depth=7,
        min_child_weight=6,
        subsample=0.85,
        colsample_bytree=0.85,
        gamma=0.2,
        reg_alpha=0.4,
        reg_lambda=3.0,
        objective='reg:absoluteerror',
        tree_method='gpu_hist',
        predictor='gpu_predictor',
        random_state=seed
    )

def cat(seed):
    return CatBoostRegressor(
        iterations=9000,
        learning_rate=0.025,
        depth=9,
        l2_leaf_reg=7,
        loss_function='MAE',
        task_type='GPU',
        random_seed=seed,
        verbose=False
    )

def lgb(seed):
    return LGBMRegressor(
        n_estimators=8000,
        learning_rate=0.025,
        num_leaves=64,
        max_depth=-1,
        subsample=0.85,
        colsample_bytree=0.85,
        objective='mae',
        device='gpu',
        random_state=seed
    )


In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
seeds = [42, 202, 777]

oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_lgb = np.zeros(len(X))

test_xgb = np.zeros(len(test_fe))
test_cat = np.zeros(len(test_fe))
test_lgb = np.zeros(len(test_fe))


In [ ]:
for seed in seeds:
    for fold, (tr, val) in enumerate(kf.split(X), 1):
        print(f"Seed {seed} | Fold {fold}")

        m1 = xgb(seed)
        m1.fit(X.iloc[tr], y_log[tr],
               eval_set=[(X.iloc[val], y_log[val])],
               early_stopping_rounds=400,
               verbose=False)
        oof_xgb[val] += np.expm1(m1.predict(X.iloc[val])) / len(seeds)
        test_xgb += np.expm1(m1.predict(test_fe)) / (len(seeds)*kf.n_splits)

        m2 = cat(seed)
        m2.fit(X.iloc[tr], y_raw[tr],
               eval_set=(X.iloc[val], y_raw[val]),
               use_best_model=True)
        oof_cat[val] += m2.predict(X.iloc[val]) / len(seeds)
        test_cat += m2.predict(test_fe) / (len(seeds)*kf.n_splits)

        m3 = lgb(seed)
        m3.fit(X.iloc[tr], y_raw[tr])
        oof_lgb[val] += m3.predict(X.iloc[val]) / len(seeds)
        test_lgb += m3.predict(test_fe) / (len(seeds)*kf.n_splits)


In [ ]:
test_stack = np.clip(test_stack, y_min, y_max)

submission = pd.DataFrame({
    'id': test_ids,
    'yield': test_stack
})

submission.to_csv('submission.csv', index=False)
submission.head()